# Deep Network Development, 2026 Autumn - Week 9 Practice - **Exercise sheet**

## Exercises

---

### Exercise 9.1: Species classification with categorical inputs and Dropout

Predict penguin species from bill length, bill depth, flipper length, and island.
Add a categorical input, train a multi-class network, and inspect a confusion matrix.


#### Provided setup

Download the original [Palmer Penguins CSV](https://allisonhorst.github.io/palmerpenguins/).
Species labels and island names are converted to integer indices below.
Inputs and labels share a fixed shuffled ordering and a 70%/15%/remaining split.

Missing measurements are filled with training-set means. The three measurements
are standardized using training statistics. Run the setup cells unchanged.


In [ ]:
import csv
import io
import urllib.request
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.set_num_threads(1)
DATA_URL = (
    'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/'
    '156daa4301838d9fdcc5b018b29b9149bf975552/inst/extdata/penguins.csv'
)
with urllib.request.urlopen(DATA_URL, timeout=30) as stream:
    records = list(csv.DictReader(io.StringIO(stream.read().decode('utf-8'))))

NUMERIC_FEATURE_NAMES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
ISLAND_NAMES = ['Biscoe', 'Dream', 'Torgersen']
SPECIES_NAMES = ['Adelie', 'Chinstrap', 'Gentoo']
island_dict = {name: i for i, name in enumerate(ISLAND_NAMES)}
species_dict = {name: i for i, name in enumerate(SPECIES_NAMES)}
NUM_CLASSES = len(SPECIES_NAMES)

measurements = np.array([
    [np.nan if row[name] == 'NA' else float(row[name]) for name in NUMERIC_FEATURE_NAMES]
    for row in records
], dtype=np.float32)
island_indices = np.array([island_dict[row['island']] for row in records], dtype=np.int64)
labels = np.array([species_dict[row['species']] for row in records], dtype=np.int64)

order = np.random.default_rng(42).permutation(len(records))
train_end = int(0.70 * len(records))
val_end = train_end + int(0.15 * len(records))
def split_rows(array):
    shuffled = array[order]
    return shuffled[:train_end], shuffled[train_end:val_end], shuffled[val_end:]

numeric_train_raw, numeric_val_raw, numeric_test_raw = split_rows(measurements)
island_train, island_val, island_test = split_rows(island_indices)
labels_train, labels_val, labels_test = split_rows(labels)

print('Numeric feature columns:', NUMERIC_FEATURE_NAMES)
print('Categorical feature: island', island_dict)
print('Label column: species', species_dict)
for name, xs, ys in [('Training', numeric_train_raw, labels_train),
                     ('Validation', numeric_val_raw, labels_val),
                     ('Test', numeric_test_raw, labels_test)]:
    print(f'{name}: measurements {xs.shape}, labels {ys.shape}')
# Expected split sizes: 240, 51, 53.


In [ ]:
# Impute measurements using training means.
fill_values = np.nanmean(numeric_train_raw, axis=0)
filled_splits = tuple(np.where(np.isnan(xs), fill_values, xs)
                      for xs in (numeric_train_raw, numeric_val_raw, numeric_test_raw))
# Standardize measurements using training statistics.
feature_means = filled_splits[0].mean(axis=0)
feature_stds = filled_splits[0].std(axis=0)
numeric_train, numeric_val, numeric_test = tuple(
    (xs - feature_means) / feature_stds for xs in filled_splits
)


#### Task 1: One-hot encoding an input feature

Implement `add_island_feature(numeric_inputs, island_indices)`. Convert island
indices to `float32` one-hot rows in the order of `ISLAND_NAMES`, append them to
the three measurement columns, and return an array of shape `(N, 6)`.
Keep the original arrays unchanged.


In [ ]:
def add_island_feature(numeric_inputs, island_indices):
    # TODO: one-hot encode the island and append its columns to the measurements.
    # SOLUTION 1 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 1 END

inputs_train = add_island_feature(numeric_train, island_train)
inputs_val = add_island_feature(numeric_val, island_val)
inputs_test = add_island_feature(numeric_test, island_test)
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + ['island_' + name for name in ISLAND_NAMES]

toy_numeric = np.array([[1., 2., 3.], [4., 5., 6.]], dtype=np.float32)
toy_islands = np.array([0, 2])
expected = np.array([[1., 2., 3., 1., 0., 0.],
                     [4., 5., 6., 0., 0., 1.]], dtype=np.float32)
assert np.array_equal(add_island_feature(toy_numeric, toy_islands), expected)
for numeric, islands, xs in [(numeric_train, island_train, inputs_train),
                             (numeric_val, island_val, inputs_val),
                             (numeric_test, island_test, inputs_test)]:
    assert xs.shape == (len(numeric), 6) and xs.dtype == np.float32
    assert np.array_equal(xs[:, :3], numeric)
    assert np.array_equal(xs[:, 3:].argmax(axis=1), islands)
    assert np.all(xs[:, 3:].sum(axis=1) == 1) and np.isfinite(xs).all()
print('Feature columns:', FEATURE_NAMES)
print('First three training inputs:')
print(inputs_train[:3])
print('Task 1 passed!')


#### Task 2: Dataset and DataLoader

Complete `ClassificationDataset`. Store inputs and labels as tensors with
appropriate dtypes. Return one input vector and one scalar label from `__getitem__`.

Create the Datasets, then the three DataLoaders with batch size 32. Shuffle training and validation,
but keep test order fixed. Batch shapes must be `(B, 6)` and `(B,)`.


In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, input_split, labels_split):
        # TODO: create input and target tensors with appropriate dtypes.
        # SOLUTION 2 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 2 END

    def __len__(self):
        return self.xs.shape[0]

    def __getitem__(self, idx):
        # TODO: return one input vector and one scalar class index.
        # SOLUTION 3 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 3 END

BATCH_SIZE = 32

# TODO: create the three Datasets and DataLoaders with the requested shuffling.
# SOLUTION 4 START
raise NotImplementedError("Complete this part")
# SOLUTION 4 END

sample_x, sample_y = train_dataset[0]
assert sample_x.shape == (6,) and sample_x.dtype == torch.float32
assert sample_y.shape == () and sample_y.dtype == torch.long
for name, loader in [('Training', dataloader_train),
                     ('Validation', dataloader_val), ('Test', dataloader_test)]:
    batch_xs, batch_ys = next(iter(loader))
    assert batch_xs.shape == (BATCH_SIZE, 6) and batch_ys.shape == (BATCH_SIZE,)
    assert batch_ys.dtype == torch.long
    print(f'{name} batch shapes:', batch_xs.shape, batch_ys.shape)
print('Task 2 passed!')


#### Task 3: A multi-class network with Dropout

Implement and instantiate `SpeciesNet` as `model`, using Tanh activation
functions and Dropout with probability 0.2.


In [ ]:
torch.manual_seed(7)

class SpeciesNet(nn.Module):
    def __init__(self, n_input_vars):
        # TODO: implement the initialization.
        # SOLUTION 5 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 5 END

    def forward(self, x):
        # TODO: implement the forward pass.
        # SOLUTION 6 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 6 END

# TODO: instantiate the model.
# SOLUTION 7 START
raise NotImplementedError("Complete this part")
# SOLUTION 7 END

modules = list(model.modules())
assert any(isinstance(module, nn.Tanh) for module in modules)
dropout_layers = [module for module in modules if isinstance(module, nn.Dropout)]
assert dropout_layers and all(module.p == 0.2 for module in dropout_layers)
assert model(train_dataset.xs[:5]).shape == (5, NUM_CLASSES)
print('Task 3 network passed!')


In [ ]:
# Provided: inspect Dropout in both modes, without recording gradients.
probe_x = train_dataset.xs[:8]
with torch.no_grad():
    model.train()
    train_probe_1, train_probe_2 = model(probe_x), model(probe_x)
    model.eval()
    eval_probe_1, eval_probe_2 = model(probe_x), model(probe_x)
print('Training predictions identical:', torch.equal(train_probe_1, train_probe_2))
print('Evaluation predictions identical:', torch.equal(eval_probe_1, eval_probe_2))
assert not torch.equal(train_probe_1, train_probe_2)
assert torch.equal(eval_probe_1, eval_probe_2)


#### Task 4: Classification accuracy

Implement `classification_accuracy(logits, targets)`, returning the fraction of
correct predictions as a scalar PyTorch tensor.


In [ ]:
@torch.no_grad()
def classification_accuracy(logits, targets):
    assert logits.shape == (len(targets), NUM_CLASSES) and targets.ndim == 1
    # TODO: calculate the fraction of correct predictions.
    # SOLUTION 8 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 8 END

toy_logits = torch.tensor([[4., 1., 0.], [0., 3., 1.], [2., 0., 4.], [1., 5., 2.]])
toy_targets = torch.tensor([0, 2, 2, 1])
assert torch.isclose(classification_accuracy(toy_logits, toy_targets), torch.tensor(0.75))
assert classification_accuracy(toy_logits, toy_targets).ndim == 0
print('Task 4 passed!')


#### Task 5: Training and validation

Complete prediction and loss calculation, backpropagation and parameter updates,
and the training and validation modes. Choose an appropriate loss.

In [ ]:
n_epochs = 50
learning_rate = 0.001
# TODO: choose the loss for this network and integer class labels.
# SOLUTION 9 START
raise NotImplementedError("Complete this part")
# SOLUTION 9 END
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
from copy import deepcopy

patience = 5
best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_state = deepcopy(model.state_dict())
train_losses, val_losses = [], []
train_accuracy, val_accuracy = [], []

for epoch_idx in range(n_epochs):
    # TODO: select the model mode for learning with Dropout
    # SOLUTION 10 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 10 END
    train_losses_in_epoch, train_accs_in_epoch = [], []

    for train_batch_xs, train_batch_ys in dataloader_train:
        optimizer.zero_grad()
        # TODO: make predictions for this batch and calculate its loss
        # SOLUTION 11 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 11 END
        acc = classification_accuracy(train_batch_logits, train_batch_ys)
        train_losses_in_epoch.append(loss.detach())
        train_accs_in_epoch.append(acc)
        # TODO: backpropagate and then update the model parameters
        # SOLUTION 12 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 12 END

    # Average batch means
    avg_tr_epoch_loss = torch.stack(train_losses_in_epoch).mean()
    avg_tr_epoch_acc = torch.stack(train_accs_in_epoch).mean()
    train_losses.append(avg_tr_epoch_loss.item())
    train_accuracy.append(avg_tr_epoch_acc.item())

    # TODO: select the model mode for reliable validation with Dropout
    # SOLUTION 13 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 13 END
    val_losses_in_epoch, val_accs_in_epoch = [], []
    for val_batch_xs, val_batch_ys in dataloader_val:
        # TODO: calculate validation logits and loss without recording gradients.
        # SOLUTION 14 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 14 END
        acc = classification_accuracy(val_batch_logits, val_batch_ys)
        val_losses_in_epoch.append(loss)
        val_accs_in_epoch.append(acc)

    avg_val_epoch_loss = torch.stack(val_losses_in_epoch).mean()
    avg_val_epoch_acc = torch.stack(val_accs_in_epoch).mean()
    val_losses.append(avg_val_epoch_loss.item())
    val_accuracy.append(avg_val_epoch_acc.item())

    if (epoch_idx + 1) % 10 == 0:
        print(f'Epoch #{epoch_idx + 1}, tr loss: {avg_tr_epoch_loss:.4f}, '
              f'val loss: {avg_val_epoch_loss:.4f}, tr acc: {avg_tr_epoch_acc:.3f}, '
              f'val acc: {avg_val_epoch_acc:.3f}')

    # Early stopping.
    if avg_val_epoch_loss.item() < best_val_loss:
        best_val_loss = avg_val_epoch_loss.item()
        epochs_without_improvement = 0
        best_model_state = deepcopy(model.state_dict())
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
        print(f'Early stopping at epoch {epoch_idx + 1}.')
        break

model.load_state_dict(best_model_state)
assert 0 < len(train_losses) == len(val_losses) <= n_epochs
assert np.isfinite(train_losses).all() and np.isfinite(val_losses).all()
assert len(train_accuracy) == len(val_accuracy) == len(train_losses)
assert all(not loss.requires_grad for loss in train_losses_in_epoch + val_losses_in_epoch)
print('Training finished.')


In [ ]:
# Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(train_losses, label='Training')
axes[0].plot(val_losses, label='Validation')
axes[0].set(xlabel='Epochs', ylabel='Cross-entropy loss', title='Loss', ylim=(0, None))
axes[1].plot(train_accuracy, label='Training')
axes[1].plot(val_accuracy, label='Validation')
axes[1].set(xlabel='Epochs', ylabel='Accuracy', title='Accuracy', ylim=(0, 1.05))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


#### Task 6: Confusion matrix

Implement `confusion_matrix(logits, targets, num_classes)`, returning a NumPy
integer array. Rows represent true classes and columns represent predicted classes.
Count every example. Test predictions and plotting are supplied.


In [ ]:
# TODO: implement the confusion matrix
# SOLUTION 15 START
def confusion_matrix(logits, targets, num_classes):
    raise NotImplementedError("Complete this part")
# SOLUTION 15 END

# Includes a repeated true/predicted pair and a class that is never predicted.
toy_logits = torch.tensor([[3., 1., 0.], [4., 2., 1.], [1., 0., 3.],
                           [0., 1., 4.], [5., 1., 0.], [3., 0., 1.]])
toy_targets = torch.tensor([0, 1, 1, 2, 2, 2])
expected = np.array([[1, 0, 0], [1, 0, 1], [2, 0, 1]])
assert np.array_equal(confusion_matrix(toy_logits, toy_targets, 3), expected)

# Provided: collect matching predictions and labels over the full test set.
model.eval()
test_logits, test_targets = [], []
with torch.no_grad():
    for test_batch_xs, test_batch_ys in dataloader_test:
        test_logits.append(model(test_batch_xs))
        test_targets.append(test_batch_ys)
test_logits = torch.cat(test_logits, dim=0)
test_targets = torch.cat(test_targets, dim=0)
test_loss = loss_fn(test_logits, test_targets)
test_accuracy = classification_accuracy(test_logits, test_targets)
test_matrix = confusion_matrix(test_logits, test_targets, NUM_CLASSES)

assert test_matrix.shape == (NUM_CLASSES, NUM_CLASSES)
assert np.issubdtype(test_matrix.dtype, np.integer)
assert test_matrix.sum() == len(test_dataset)
assert np.array_equal(test_matrix.sum(axis=1), np.bincount(labels_test, minlength=NUM_CLASSES))
assert test_matrix.trace() == (test_logits.argmax(dim=1) == test_targets).sum().item()
print(f'Test loss: {test_loss:.4f}, test accuracy: {test_accuracy:.3f}')
print('Confusion matrix (rows=true, columns=predicted):')
print(test_matrix)
print('Task 6 passed!')


In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(test_matrix, cmap='Blues', vmin=0)
ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
       xticklabels=SPECIES_NAMES, yticklabels=SPECIES_NAMES,
       xlabel='Predicted species', ylabel='True species', title='Test confusion matrix')
for row in range(NUM_CLASSES):
    for col in range(NUM_CLASSES):
        ax.text(col, row, str(test_matrix[row, col]), ha='center', va='center',
                color='white' if test_matrix[row, col] > test_matrix.max() / 2 else 'black')
fig.colorbar(image, ax=ax, label='Examples')
plt.tight_layout()
plt.show()
